In [1]:
import numpy as np, time
from sklearn.naive_bayes import CategoricalNB

DATA = "data"
FEAT = [150, 150, 8, 8, 19, 10, 10]

X = np.load(f"{DATA}/preprocessed/train_X.npy")
y = np.load(f"{DATA}/preprocessed/train_y.npy")
e = np.load(f"{DATA}/preprocessed/train_exist.npy").astype(bool)
X_te = np.load(f"{DATA}/preprocessed/test_X.npy")[:50000]
e_te = np.load(f"{DATA}/preprocessed/test_exist.npy")[:50000].astype(bool)
print(f"train: {len(X)} pairs, test: {len(X_te)}")


train: 10380088 pairs, test: 50000


In [2]:
# accuracy at different training sizes
for pct in [1, 5, 10, 25, 50, 100]:
    n = int(len(X) * pct / 100)
    ec = CategoricalNB(alpha=1.0, min_categories=2)
    ec.fit(X[:n], e[:n])
    pc = CategoricalNB(alpha=1.0, min_categories=FEAT)
    pc.fit(X[:n][e[:n]], y[:n][e[:n]])
    acc = pc.score(X_te[e_te], y[:len(X_te)][e_te])
    print(f"{pct:3d}% ({n//1000}k): acc={acc:.3f}")


  1% (103k): acc=0.006
  5% (519k): acc=0.004


 10% (1038k): acc=0.005


 25% (2595k): acc=0.006


 50% (5190k): acc=0.006


100% (10380k): acc=0.006


In [3]:
# flat recall at different sizes
for pct in [10, 25, 50, 100]:
    n = int(len(X) * pct / 100)
    ec = CategoricalNB(alpha=1.0, min_categories=2)
    ec.fit(X[:n], e[:n])
    pc = CategoricalNB(alpha=1.0, min_categories=FEAT)
    pc.fit(X[:n][e[:n]], y[:n][e[:n]])
    el = ec.predict_log_proba(X_te)[:, 1]
    pl = pc.predict_log_proba(X_te)
    best = np.argmax(pl, axis=1)
    c = pl[np.arange(len(X_te)), best]
    comb = el + c
    top100 = np.argsort(comb)[::-1][:100]
    hits = sum(1 for i in top100 if e_te[i])
    print(f"{pct:3d}%: top-100 hits={hits}/{e_te.sum()} ({100*hits/e_te.sum():.1f}%)")


 10%: top-100 hits=37/1587 (2.3%)


 25%: top-100 hits=30/1587 (1.9%)


 50%: top-100 hits=28/1587 (1.8%)


100%: top-100 hits=24/1587 (1.5%)
